# Predictive Inference Model

This will be used after we establish all proper parameters for the distributions in `study_habits_eda.ipynb`, which is where the distribution is generated and then exported to `base_popularity_distributions.csv`.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# How Often In Advance People Study

In [3]:
df_melted = pd.read_csv('./data/base_popularity_distributions.csv')

In [4]:
# Calculate study pressure
import math

def get_study_pressure_multiplier(exams_list):
    """
    exams_list: A list of integers representing 'days away' for each exam.
    e.g., [1, 3] means one exam tomorrow, one exam in 3 days.
    (Remember to pass CBTF exams based only on their final day).
    """
    
    # 1. Calculate Raw Score
    raw_score = 0
    for days_away in exams_list:
        if 0 <= days_away <= 3:
            raw_score += (4 - days_away) 
            
    # 2. Tuning Parameters
    THRESHOLD = 4.0      # E.g., 1 exam today (4pts) = no effect. Must exceed 4.
    MAX_BOOST = 0.60     # The multiplier approaches, but never exceeds, 1.60x
    GROWTH_RATE = 0.25   # Controls how fast the curve hits the ceiling
    
    # 3. Apply Threshold and Squash
    if raw_score <= THRESHOLD:
        return 1.0  # Normal baseline demand
    else:
        # Exponential curve that flatlines at the upper bound
        boost = MAX_BOOST * (1 - math.exp(-GROWTH_RATE * (raw_score - THRESHOLD)))
        return 1.0 + boost

# --- Example Scenarios ---
# Scenario A: 1 exam tomorrow (3 pts).
print(get_study_pressure_multiplier([1])) 
# Output: 1.0 (Doesn't beat the threshold)

# Scenario B: Midterm season. 1 exam tmrw, 1 in 2 days, 1 in 3 days (3+2+1 = 6 pts)
print(get_study_pressure_multiplier([1, 2, 3])) 
# Output: ~1.23 (Demand increases by 23%)

# Scenario C: Hell week. 3 exams tomorrow, 2 the next day (3+3+3+2+2 = 13 pts)
print(get_study_pressure_multiplier([1, 1, 1, 2, 2])) 
# Output: ~1.53 (Demand increases by 53%, safely below the 1.60x cap)

1.0
1.2360816041724199
1.5367604652628812


In [5]:
import math
import numpy as np

class CampusDemandModel:
    def __init__(self, baseline_df):
        # baseline_df is the dataframe we generated in the previous step 
        # (contains Day, Hour, Building, Base_Popularity)
        self.baseline = baseline_df
        
        self.days_of_week = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        
        # Survey votes for active distribution (excluding Dorms)
        self.building_votes = {
            'Grainger Library': 15, 'CIF': 14, 'Illini Union': 13, 
            'Main Library': 10, 'Funk Library': 10, 'BIF': 7, 
            'Siebel Center for Design': 7, 'Siebel Center for CS': 5
        }
        self.total_votes = sum(self.building_votes.values())
        
        # Structured Exam Schedule (Exams mapped to the day they end)
        # Format: (Week, Day) : Number of exams
        self.exam_schedule = {
            # Week 3
            (3, 'Tuesday'): 1,   # Chem 104 CBTF
            (3, 'Thursday'): 1,  # CS 173 CBTF
            (3, 'Friday'): 1,    # Ochem 1
            # Week 4
            (4, 'Wednesday'): 2, # Math 257 CBTF, Chem 101
            (4, 'Thursday'): 1,  # CS 173 CBTF
            (4, 'Friday'): 1,    # Math 241 A/C CBTF
            (4, 'Saturday'): 1,  # MCB 150 CBTF
            # Week 5
            (5, 'Tuesday'): 2,   # Math 220, Chem 104 CBTF
            (5, 'Wednesday'): 1, # Chem 102
            (5, 'Thursday'): 2,  # Math 241 B, CS 173 CBTF
            (5, 'Friday'): 1,    # Phys 211
            (5, 'Sunday'): 1,    # Phys 212 CBTF
            # Week 6
            (6, 'Thursday'): 1,  # CS 173 CBTF
            (6, 'Friday'): 1,    # CS 173 C
            # Week 7
            (7, 'Tuesday'): 1,   # Chem 104 CBTF
            (7, 'Wednesday'): 1, # Chem 101
            (7, 'Thursday'): 1,  # CS 173 CBTF
            (7, 'Friday'): 2,    # Math 241 A/C CBTF, Ochem 1
            # Week 8
            (8, 'Wednesday'): 1, # Math 257 CBTF
            (8, 'Thursday'): 1,  # CS 173 CBTF
            (8, 'Friday'): 1,    # MCB 150 CBTF
            # Week 9
            (9, 'Tuesday'): 2,   # Math 220, Chem 104 CBTF
            (9, 'Thursday'): 2,  # Math 241 B, CS 173 CBTF
            (9, 'Friday'): 2,    # Math 241 A/C CBTF, Phys 211
            # Week 10
            (10, 'Wednesday'): 1, # Chem 102
            (10, 'Thursday'): 1,  # CS 173 CBTF
            (10, 'Friday'): 1,    # CS 173 C
            (10, 'Sunday'): 1,    # Phys 212 CBTF
            # ... (You can continue populating Weeks 11-15 here)
        }

    def _get_future_day(self, current_week, current_day, days_ahead):
        """Helper to find the (week, day) for X days from now."""
        idx = self.days_of_week.index(current_day) + days_ahead
        new_week = current_week + (idx // 7)
        new_day = self.days_of_week[idx % 7]
        return new_week, new_day

    def _calculate_pressure(self, week, day):
        """Looks ahead 3 days, fetches exams, and returns the multiplier."""
        exams_list = []
        
        # Look at Today (0), Tomorrow (1), In 2 days (2), In 3 days (3)
        for days_ahead in range(4):
            target_week, target_day = self._get_future_day(week, day, days_ahead)
            exam_count = self.exam_schedule.get((target_week, target_day), 0)
            
            # Add an entry to the list for EVERY exam found that day
            exams_list.extend([days_ahead] * exam_count)
            
        # The squashing logic we built earlier
        raw_score = sum([(4 - d) for d in exams_list])
        THRESHOLD, MAX_BOOST, GROWTH_RATE = 4.0, 0.60, 0.25
        
        if raw_score <= THRESHOLD:
            return 1.0
        return 1.0 + (MAX_BOOST * (1 - math.exp(-GROWTH_RATE * (raw_score - THRESHOLD))))

    def predict_demand(self, week, day, hour):
        """The main prediction pipeline."""
        
        # 1. Get the Pressure Multiplier
        pressure_multiplier = self._calculate_pressure(week, day)
        
        # 2. Fetch Baseline for this Day/Hour
        mask = (self.baseline['Day'] == day) & (self.baseline['Hour'] == hour)
        current_hour_df = self.baseline[mask].copy()
        
        if current_hour_df.empty:
            return "Invalid time parameters."

        # 3. Apply Multiplier & Calculate Raw Demand
        # Note: If base popularity is 0 (closed), demand stays 0
        current_hour_df['Raw_Demand'] = current_hour_df['Base_Popularity'] * pressure_multiplier
        
        # 4. Calculate Spillover
        raw_spillover = 0.0
        for idx, row in current_hour_df.iterrows():
            if row['Raw_Demand'] > 1.0:
                raw_spillover += (row['Raw_Demand'] - 1.0)
                current_hour_df.at[idx, 'Raw_Demand'] = 1.0 # Cap the building at 100%
        
        # 5. Route the Spillover (Reduced by 70% for people giving up/going to dorms)
        retained_spillover = raw_spillover * 0.30
        
        if retained_spillover > 0:
            for idx, row in current_hour_df.iterrows():
                b_name = row['Building']
                # Only distribute to buildings that are OPEN (Base > 0) and not full
                if row['Base_Popularity'] > 0 and row['Raw_Demand'] < 1.0:
                    weight = self.building_votes[b_name] / self.total_votes
                    share_of_spillover = retained_spillover * weight
                    
                    # Add spillover and enforce the 1.0 cap one last time
                    new_demand = min(1.0, row['Raw_Demand'] + share_of_spillover)
                    current_hour_df.at[idx, 'Raw_Demand'] = new_demand

        # Return results neatly
        results = current_hour_df[['Building', 'Raw_Demand']].set_index('Building').to_dict()['Raw_Demand']
        return {
            'Pressure_Multiplier': round(pressure_multiplier, 3),
            'Location_Demands': {k: round(v, 3) for k, v in results.items()}
        }

# ==========================================
# Example Usage
# ==========================================
# (Assuming df_melted is available from the previous step)
model = CampusDemandModel(baseline_df=df_melted)
print(model.predict_demand(week=5, day='Thursday', hour=14))

{'Pressure_Multiplier': 1.519, 'Location_Demands': {'Grainger Library': 1.0, 'Funk Library': 1.0, 'Main Library': 1.0, 'Illini Union': 0.794, 'CIF': 1.0, 'BIF': 0.988, 'Siebel Center for CS': 0.306, 'Siebel Center for Design': 0.935}}


In [6]:
df_melted.sample(n=10)

,Day,Hour,Building,Base_Popularity
619,Friday,19,Illini Union,0.662172
828,Sunday,12,CIF,0.296697
334,Sunday,22,Funk Library,0.008570
772,Friday,4,CIF,0.000000
1003,Sunday,19,BIF,0.060555
936,Friday,0,BIF,0.000000
1042,Tuesday,10,Siebel Center for CS,0.186625
556,Wednesday,4,Illini Union,0.000000
88,Thursday,16,Grainger Library,0.864005
393,Wednesday,9,Main Library,0.324403


In [7]:
df_melted[(df_melted['Day'] == 'Tuesday') & (df_melted['Hour'] >= 10) & (df_melted['Hour'] <= 20)]

,Day,Hour,Building,Base_Popularity
34,Tuesday,10,Grainger Library,0.275743
35,Tuesday,11,Grainger Library,0.398144
36,Tuesday,12,Grainger Library,0.529816
37,Tuesday,13,Grainger Library,0.649765
38,Tuesday,14,Grainger Library,0.734404
...,...,...,...,...
1216,Tuesday,16,Siebel Center for Design,0.476439
1217,Tuesday,17,Siebel Center for Design,0.360886
1218,Tuesday,18,Siebel Center for Design,0.244612
1219,Tuesday,19,Siebel Center for Design,0.148365


In [10]:
model.predict_demand(week=3, day='Wednesday', hour=12)

{'Pressure_Multiplier': 1.133,
 'Location_Demands': {'Grainger Library': 0.706,
  'Funk Library': 0.911,
  'Main Library': 0.866,
  'Illini Union': 0.906,
  'CIF': 0.659,
  'BIF': 0.606,
  'Siebel Center for CS': 0.349,
  'Siebel Center for Design': 0.638}}